# Single-Node Profiling - Mamba-130M

Visualizes the single-node baseline benchmarks collected by `scripts/profile_single_node.py`.
These numbers are the reference against which multi-node pipeline parallelism overhead is measured.

## Data Source

The CSV at `benchmarks/single_node_baseline.csv` is generated by running:

```bash
python scripts/profile_single_node.py
```

Each row records latency statistics (mean, median, min, max, p95) and peak memory for a
specific token count and pass type (forward pass or generation).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## Load Data

In [ ]:
df = pd.read_csv("../benchmarks/single_node_baseline.csv")
df

## Latency vs. Sequence Length

Mamba is O(n) in sequence length. This chart shows whether the measured latency
scales linearly with token count for both forward pass and generation.

In [ ]:
fig, ax = plt.subplots()

for pass_type, group in df.groupby("pass_type"):
    group = group.sort_values("tokens")
    ax.plot(
        group["tokens"],
        group["mean_ms"],
        marker="o",
        linewidth=2,
        label=pass_type,
    )
    ax.fill_between(
        group["tokens"],
        group["min_ms"],
        group["max_ms"],
        alpha=0.15,
    )

ax.set_xlabel("Input Tokens")
ax.set_ylabel("Latency (ms)")
ax.set_title("Mamba-130M Single-Node Latency vs. Sequence Length")
ax.legend()
plt.tight_layout()
plt.show()

## Latency Breakdown by Pass Type

Side-by-side comparison of forward pass vs. generation latency at each token count.
The difference is the autoregressive generation overhead (30 new tokens).

In [ ]:
pivot = df.pivot(index="tokens", columns="pass_type", values="mean_ms")

fig, ax = plt.subplots()
pivot.plot(kind="bar", ax=ax, width=0.7)

ax.set_xlabel("Input Tokens")
ax.set_ylabel("Mean Latency (ms)")
ax.set_title("Forward Pass vs. Generation Latency")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## Peak Memory Usage

Peak RSS memory during inference. On a 4GB Raspberry Pi 5, available memory
for the model + inference is roughly 3.5GB after OS overhead.

In [ ]:
fig, ax = plt.subplots()

for pass_type, group in df.groupby("pass_type"):
    group = group.sort_values("tokens")
    ax.plot(
        group["tokens"],
        group["peak_memory_mb"],
        marker="s",
        linewidth=2,
        label=pass_type,
    )

ax.set_xlabel("Input Tokens")
ax.set_ylabel("Peak Memory (MB)")
ax.set_title("Peak Memory Usage vs. Sequence Length")
ax.legend()
plt.tight_layout()
plt.show()

## Latency Variability (P95 vs. Median)

Shows how consistent the latency is across runs. A large gap between
median and P95 indicates jitter from GC, OS scheduling, or thermal throttling.

In [ ]:
fig, ax = plt.subplots()

for pass_type, group in df.groupby("pass_type"):
    group = group.sort_values("tokens")
    ax.plot(
        group["tokens"],
        group["median_ms"],
        marker="o",
        linewidth=2,
        label=f"{pass_type} (median)",
    )
    ax.plot(
        group["tokens"],
        group["p95_ms"],
        marker="^",
        linewidth=2,
        linestyle="--",
        label=f"{pass_type} (p95)",
    )

ax.set_xlabel("Input Tokens")
ax.set_ylabel("Latency (ms)")
ax.set_title("Latency Variability: Median vs. P95")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Summary Table

In [ ]:
summary = df[[
    "hostname", "tokens", "pass_type",
    "mean_ms", "median_ms", "p95_ms", "peak_memory_mb"
]].copy()
summary.columns = ["Host", "Tokens", "Pass Type", "Mean (ms)", "Median (ms)", "P95 (ms)", "Peak Mem (MB)"]
summary